In [0]:

# Databricks widgets: 분기(스냅샷 분기)와 프롬프트 버전 외부 주입
dbutils.widgets.text("snapshot_quarter", "2026Q1")
dbutils.widgets.text("prompt_version", "v1.0.0")
dbutils.widgets.text("top_k", "3")  # 상/하위 driver Top-K
dbutils.widgets.text("llm_model", "databricks-gpt-5-2")  # OpenAI 모델명 (예시)
dbutils.widgets.text("openai_secret_scope", "kv") # Databricks Secret Scope
dbutils.widgets.text("openai_secret_key", "openai_api_key") # Secret key name

SNAPSHOT_QUARTER = dbutils.widgets.get("snapshot_quarter")
PROMPT_VERSION   = dbutils.widgets.get("prompt_version")
TOP_K            = int(dbutils.widgets.get("top_k"))
LLM_MODEL        = dbutils.widgets.get("llm_model")
SECRET_SCOPE     = dbutils.widgets.get("openai_secret_scope")
SECRET_KEY       = dbutils.widgets.get("openai_secret_key")

print(f"[CONFIG] snapshot_quarter={SNAPSHOT_QUARTER}, prompt_version={PROMPT_VERSION}, top_k={TOP_K}, model={LLM_MODEL}")


In [0]:
# PySpark
from pyspark.sql import functions as F
from pyspark.sql import Window as W
from pyspark.sql.types import *

# Python
import os, json, time, math, traceback
from datetime import datetime

import mlflow
mlflow.openai.autolog()

# OpenAI (python SDK 1.x 스타일)
try:
    from openai import OpenAI
except ImportError:
    # Databricks 환경에 설치되어 있지 않다면 다음 코멘트를 참고하세요.
    # %pip install openai==1.*
    raise

# OpenAI API Key 로딩 (우선 순위: Secret → env)
OPENAI_API_KEY = None
try:
    OPENAI_API_KEY = dbutils.secrets.get(SECRET_SCOPE, SECRET_KEY)
except Exception:
    OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")

if not OPENAI_API_KEY:
    raise RuntimeError("OpenAI API Key가 설정되지 않았습니다. Databricks Secret 또는 환경변수를 확인해주세요.")

client = OpenAI(api_key=OPENAI_API_KEY)

In [0]:

model_summary_sdf = spark.table("sandbox.z_jungryo_lee.voc_wls_model_summary")
coef_summary_sdf  = spark.table("sandbox.z_jungryo_lee.voc_wls_coef_summary")

print("[INFO] model_summary count:", model_summary_sdf.count())
print("[INFO] coef_summary count :", coef_summary_sdf.count())


In [0]:
# Driver 필터
drivers_sdf = (
    coef_summary_sdf
      .withColumn("abs_coef", F.abs(F.col("coefficient")))
      .filter( (F.col("p_value") < F.lit(0.1)) & (F.col("abs_coef") >= F.lit(0.07)) )
)

# 랭킹 윈도우
wpos = W.partitionBy("group_dim", "group_key", "y_feature").orderBy(F.col("coefficient").desc())
wneg = W.partitionBy("group_dim", "group_key", "y_feature").orderBy(F.col("coefficient").asc())

ranked_sdf = (
    drivers_sdf
      .withColumn("rank_pos", F.when(F.col("coefficient") > 0, F.dense_rank().over(wpos)))
      .withColumn("rank_neg", F.when(F.col("coefficient") < 0, F.dense_rank().over(wneg)))
)

top_pos_sdf = (
    ranked_sdf
      .filter(F.col("coefficient") > 0)
      .filter(F.col("rank_pos") <= F.lit(TOP_K))
      .select("group_dim","group_key","y_feature","x_feature","coefficient")
)

top_neg_sdf = (
    ranked_sdf
      .filter(F.col("coefficient") < 0)
      .filter(F.col("rank_neg") <= F.lit(TOP_K))
      .select("group_dim","group_key","y_feature","x_feature","coefficient")
)


In [0]:
model_summary_sdf

In [0]:
# 모델 요약(모든 group_key 포함) 집계
ms_agg_sdf = (
    model_summary_sdf
      .groupBy("group_dim","y_feature")
      .agg(F.collect_list(F.struct(
            F.col("group_key"),
            F.col("R_squared"),
            F.col("Adj_R_squared"),
            F.col("F_statistic"),
            F.col("Prob_F").alias("model_p_value"),
            F.col("AIC"),
            F.col("BIC"),
            F.col("Cond_No"),
            F.col("y_obs"),
      )).alias("model_stats"))
)

# 필터된 드라이버 전체(조건 충족)
drivers_agg_sdf = (
    ranked_sdf
      .select("group_dim","y_feature","group_key","x_feature","coefficient","p_value","t_value","x_obs")
      .groupBy("group_dim","y_feature")
      .agg(F.collect_list(F.struct(
            F.col("group_key"),
            F.col("x_feature"),
            F.col("coefficient"),
            F.col("p_value"),
            F.col("t_value"),
            F.col("x_obs")
      )).alias("drivers_filtered"))
)

# Top-K positive/negative 수집
pos_agg_sdf = (
    top_pos_sdf
      .groupBy("group_dim","y_feature")
      .agg(F.collect_list(F.struct(
            F.col("group_key"),
            F.col("x_feature"),
            F.col("coefficient")
      )).alias("top_positive_drivers"))
)

neg_agg_sdf = (
    top_neg_sdf
      .groupBy("group_dim","y_feature")
      .agg(F.collect_list(F.struct(
            F.col("group_key"),
            F.col("x_feature"),
            F.col("coefficient")
      )).alias("top_negative_drivers"))
)

# 전체 조인 (겹치는 키는 (group_dim, y_feature))
payload_sdf = (
    ms_agg_sdf
      .join(drivers_agg_sdf, ["group_dim","y_feature"], "left")
      .join(pos_agg_sdf,    ["group_dim","y_feature"], "left")
      .join(neg_agg_sdf,    ["group_dim","y_feature"], "left")
)

print("[INFO] payload groups:", payload_sdf.count())

In [0]:
# 레이트 리밋: 요청 간 최소 간격(초)
RATE_LIMIT_SECONDS = 0.6

# 재시도
MAX_RETRIES = 4
BACKOFF_BASE = 1.8  # 지수 백오프 배수
REQUEST_TIMEOUT = 60  # 초

SYSTEM_PROMPT = """You are an expert product planning analyst for TV products.
Return ONLY strict JSON. No preface, no markdown.
Analyze the payload and produce actionable, comparative insights when group_dim is 'brand' or 'country', or market-wide insights when 'all'.
Use concise, decision-oriented language."""

# 사용자 프롬프트 템플릿
def build_user_prompt(case_hint: str):
    return f"""
You will receive a JSON payload. Follow STRICTLY:

- Return ONLY JSON matching this schema:

{{
  "headline": "One-line summary",
  "summary": "2~4 sentence product-planning insight",
  "top_takeaways": ["key point 1","key point 2","key point 3"],
  "risk_signal": "risk or warning interpretation",
  "recommended_actions": ["action 1","action 2","action 3"],
  "top_positive_drivers": [{{"group_key":"...", "x_feature":"...", "coef":0.0}}],
  "top_negative_drivers": [{{"group_key":"...", "x_feature":"...", "coef":-0.0}}],
  "model_comment": "interpretation of R² and review counts"
}}

- case: {case_hint}
- Strict JSON only, no extra text.
"""

def detect_case(group_dim: str) -> str:
    if group_dim == "all":
        return "overall market insight: strongest/weakest drivers, model strength."
    else:
        return "comparative insight across group_keys: differences in drivers, strongest vs weakest, negative signals."

def call_openai_with_retry(messages, model: str, temperature: float = 0.0):
    last_err = None
    for attempt in range(1, MAX_RETRIES+1):
        try:
            # Databricks 환경에서 timeout 옵션
            resp = client.chat.completions.create(
                model=model,
                messages=messages,
                temperature=temperature,
                response_format={"type": "json_object"},
                timeout=REQUEST_TIMEOUT
            )
            content = resp.choices[0].message.content
            # 엄격한 JSON 파싱
            data = json.loads(content)
            # 레이트리밋
            time.sleep(RATE_LIMIT_SECONDS)
            return data, content  # (파싱된 dict, 원문 문자열)
        except Exception as e:
            last_err = e
            wait = (BACKOFF_BASE ** (attempt-1))
            print(f"[WARN] OpenAI call failed (attempt {attempt}/{MAX_RETRIES}): {repr(e)} → wait {wait:.1f}s")
            time.sleep(wait)
    # 모든 재시도 실패 시
    raise RuntimeError(f"OpenAI call failed after {MAX_RETRIES} attempts: {repr(last_err)}")

In [0]:
rows = payload_sdf.collect()
print(f"[INFO] to-generate combinations: {len(rows)}")

results = []  # Delta에 MERGE할 최종 레코드 목록

for i, r in enumerate(rows, 1):
    try:
        group_dim = r["group_dim"]
        y_feature = r["y_feature"]
        model_stats = r["model_stats"] or []
        drivers_filtered = r.get("drivers_filtered", None)
        top_pos = r.get("top_positive_drivers", None)
        top_neg = r.get("top_negative_drivers", None)

        # Spark Row → Python dict(list)로 안전 변환
        def rows_to_pylist(lst):
            if not lst: return []
            return [dict(x.asDict()) if hasattr(x, "asDict") else dict(x) for x in lst]

        model_stats_py = rows_to_pylist(model_stats)
        drivers_filtered_py = rows_to_pylist(drivers_filtered)
        top_pos_py = rows_to_pylist(top_pos)
        top_neg_py = rows_to_pylist(top_neg)

        # 케이스 판별
        case_hint = detect_case(group_dim)

        # LLM 입력 페이로드
        payload = {
            "meta": {
                "snapshot_quarter": SNAPSHOT_QUARTER,
                "prompt_version": PROMPT_VERSION,
                "llm_model": LLM_MODEL,
                "generated_at": datetime.utcnow().isoformat(timespec='seconds') + "Z",
                "driver_thresholds": {"p_value": 0.05, "abs_coef": 0.15},
                "top_k": TOP_K,
                "case": "overall" if group_dim == "all" else "comparative"
            },
            "context": {
                "group_dim": group_dim,
                "y_feature": y_feature
            },
            "model_summary": model_stats_py,
            "drivers_filtered": drivers_filtered_py,
            "top_positive_drivers": top_pos_py,
            "top_negative_drivers": top_neg_py
        }

        messages = [
            {"role": "system", "content": SYSTEM_PROMPT},
            {"role": "user", "content": build_user_prompt(case_hint)},
            {"role": "user", "content": json.dumps(payload, ensure_ascii=False)}
        ]

        llm_json, raw_content = call_openai_with_retry(messages, model=LLM_MODEL, temperature=0.0)

        # 필수 필드 보정(없으면 기본값 채움)
        headline = llm_json.get("headline", "")
        summary  = llm_json.get("summary", "")
        top_takeaways = llm_json.get("top_takeaways", [])
        risk_signal = llm_json.get("risk_signal", "")
        recommended_actions = llm_json.get("recommended_actions", [])
        top_positive_drivers = llm_json.get("top_positive_drivers", [])
        top_negative_drivers = llm_json.get("top_negative_drivers", [])
        model_comment = llm_json.get("model_comment", "")

        results.append({
            "snapshot_quarter": SNAPSHOT_QUARTER,
            "prompt_version": PROMPT_VERSION,
            "group_dim": group_dim,
            "y_feature": y_feature,
            "headline": headline,
            "summary": summary,
            "top_takeaways_json": json.dumps(top_takeaways, ensure_ascii=False),
            "risk_signal": risk_signal,
            "recommended_actions_json": json.dumps(recommended_actions, ensure_ascii=False),
            "top_positive_drivers_json": json.dumps(top_positive_drivers, ensure_ascii=False),
            "top_negative_drivers_json": json.dumps(top_negative_drivers, ensure_ascii=False),
            "model_comment": model_comment,
            "payload_json": json.dumps(payload, ensure_ascii=False),
            "generated_at": datetime.utcnow().strftime("%Y-%m-%d %H:%M:%S")
        })

        if i % 5 == 0 or i == len(rows):
            print(f"[INFO] progress: {i}/{len(rows)} generated")

    except Exception as e:
        print(f"[ERROR] generating insight failed for ({r['group_dim']}, {r['y_feature']}): {repr(e)}")
        traceback.print_exc()